# Iter-SPIN — Step 1: iterative identification -> cumulative coupled indices

**Variant of SPIN (thesis Contribution 2).** Instead of the one-shot identification in the faithful
7B Step 1, this re-scores importance on the **progressively suppressed** model for `N_MAX` rounds:

```
for round in 1..N_MAX:
    score importance (f, p, g) on the CURRENT (partially suppressed) model
    compute coupled set  (authors' EXACT get_set_difference_mask)
    ZERO those weights in place            # suppress between rounds
    add them to the cumulative coupled set
    if round in SNAP: dump coupled_iter_n{round}.json   # ablation snapshot
```

Two properties make this clean and faithful:
- The importance metric `|W|*Σ|∇L|` and the coupling selection are **identical** to the faithful
  Step 1 (same `get_set_difference_mask`, same p=q). Only the outer loop + in-place suppression are new.
- Once a weight is zeroed, `|W|=0`, so its importance becomes 0 and it **auto-drops** from later
  rounds' top-k. Each round therefore finds *new* coupled weights among the survivors, and the
  cumulative set grows — exactly the "progressively modified model" the proposal describes.

**Ablation over n in {1,2,3,5,10} for free:** run the 10-round loop **once**; snapshot the *cumulative*
indices after rounds 1,2,3,5,10. Because zeroing is order-independent, applying a snapshot's cumulative
set to the original model reproduces that round's model. **n=1 == original SPIN** (sanity: `coupled_iter_n1.json`
should match the author-sample Step 1's `coupled_indices_authorsample.json`, 1201 for Qwen2-7B).

Then reuse the 7B pipeline **unchanged**: run Step 2 (HSIC + perplexity) once per snapshot to get the
decoupling-vs-capability curve; run Step 3+4 (generate + judge) only for the chosen best n.

Output: `coupled_iter_{MODEL_TAG}_n{1,2,3,5,10}.json` (auto-tagged per model). Sampling now matches the
authors' `datasets.shuffle(seed).select` (as in the 7B author-sample run). `MODEL_ID` is a variable —
uncomment one POC/scale model per run. Run: GPU **T4 x2**, Internet ON. ~15-20 min/round
(~2.5-3 h for 10 rounds). **Checkpointed per round** — a crash/timeout resumes.


## Step 0 — env + config

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import random, gc, time, json
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ---- model lineup: uncomment ONE per run. Outputs are auto-tagged by model name, so
#      different models write different files and never collide. ----
MODEL_ID='Qwen/Qwen2.5-3B-Instruct'     
USE_FAST=False   # authors' slow SentencePiece tokenizer. Set True for Llama-3/3.2 (fast-only tokenizer).
NSAMPLES=128
TARGET='mlp'
MAXLEN=1024
LAYER_GROUP=4
P=5e-7; Q=5e-7       # authors' Qwen2-7B extraction ratio (identical to faithful Step 1)
SEED=0
DTYPE=torch.float16
N_MAX=10             # total iterative rounds to run
SNAP=[1,2,3,5,10]    # ablation snapshots (cumulative indices dumped after these rounds)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
MODEL_TAG=MODEL_ID.split('/')[-1]   # e.g. Qwen2-7B-Instruct -> filenames tagged with this
print('model', MODEL_ID, '(tag', MODEL_TAG+')', '| group', LAYER_GROUP, '| p=q', P, '| N_MAX', N_MAX, '| snapshots', SNAP)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


model Qwen/Qwen2.5-3B-Instruct (tag Qwen2.5-3B-Instruct) | group 4 | p=q 5e-07 | N_MAX 10 | snapshots [1, 2, 3, 5, 10]


## Step 1 — data + model (identical to faithful Step 1)

In [2]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
CSV={'g':f'{DATA}/alpaca_cleaned_no_safety_train.csv',
     'f':f'{DATA}/beaver_train330k_fairness_safe_1k.csv',
     'p':f'{DATA}/beaver_train330k_privacy_safe_1k.csv'}

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'})  # cap sized for 7B on T4x2; harmless for smaller POC models
model.eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model split loaded | entry', GEN_DEV); memline('loaded')

def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]
# ---- authors' EXACT sample selection (utils.get_loaders disentangle=True): shuffle(seed).select ----
#      inp = prompt + response[:,1:] (unconditional BOS drop); loss on response only (prompt masked -100).
def make_batches(csv_path, n):
    traindata = load_dataset('csv', data_files={'train':csv_path}, split='train')
    sampled   = traindata.shuffle(seed=SEED).select(range(n))   # HF-datasets draw (bit-reproducible w/ datasets==2.18.0)
    out=[]
    for i in range(n):
        ep=tokenizer(str(sampled['prompt'][i]),   return_tensors='pt').input_ids
        er=tokenizer(str(sampled['response'][i]), return_tensors='pt').input_ids[:,1:]
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out
BATCHES={k:make_batches(CSV[k],NSAMPLES) for k in CSV}
print('batches | g',len(BATCHES['g']),'| f',len(BATCHES['f']),'| p',len(BATCHES['p']),'(author shuffle+select)')

Cloning into 'DEAN'...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model split loaded | entry cuda:0
  [loaded] GPU0 12.1/15.6 GPU1 12.7/15.6 GiB free


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

batches | g 127 | f 128 | p 128 (author shuffle+select)


## Step 2 — authors' exact selection + scoring + suppression helpers

In [3]:
names=target_names(model); nL=model.config.num_hidden_layers
GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]

# ---- authors' EXACT SPIN.py logic (verbatim, identical to faithful Step 1) ----
def get_set_difference_mask(p, q, W_metric1, W_metric2):
    top_p = int(p * W_metric1.shape[1] * W_metric1.shape[0])
    top_q = int(q * W_metric2.shape[1] * W_metric2.shape[0])
    top_p_indices = torch.topk(W_metric1.flatten(), top_p, largest=True)[1]
    top_q_indices = torch.topk(W_metric2.flatten(), top_q, largest=True)[1]
    unique_p = torch.unique(top_p_indices)
    unique_q = torch.unique(top_q_indices)
    mask_only_metric1 = ~torch.isin(unique_p, unique_q)
    mask_only_metric2 = ~torch.isin(unique_q, unique_p)
    mask_intersection = ~(torch.ones_like(unique_p).bool() & mask_only_metric1 & mask_only_metric2)
    return mask_only_metric1, mask_only_metric2, mask_intersection, unique_q

def coupled_indices_for_matrix(W_general, W_fair, W_priv):
    _, mask_only_trust1, _, unique_trust1_indices = get_set_difference_mask(P, Q, W_general, W_fair)
    _, mask_only_trust2, _, unique_trust2_indices = get_set_difference_mask(P, Q, W_general, W_priv)
    mask_trust_intersection = mask_only_trust1 & mask_only_trust2
    filtered_indices = unique_trust1_indices[mask_trust_intersection]
    return filtered_indices

def coupling_set_sizes(W_general, W_fair, W_priv):
    # true SET accounting on the CURRENT (suppressed) model -> tracks coupling-assumption collapse
    def tks(W, ratio):
        k=int(ratio*W.shape[0]*W.shape[1])
        return set(torch.unique(torch.topk(W.flatten(), k, largest=True)[1]).tolist()) if k>=1 else set()
    Sg=tks(W_general,P); Sf=tks(W_fair,Q); Sp=tks(W_priv,Q); fp=Sf & Sp
    return {'n_fp':len(fp),'n_fp_notg':len(fp-Sg)}

@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}  # zeroed weights -> imp 0
    del acc; gc.collect(); torch.cuda.empty_cache()
    return imp

def suppress_cumulative(cum):
    # zero every weight in the cumulative coupled set, in place (order-independent, idempotent)
    d=dict(model.named_parameters()); nz=0
    for n,idxset in cum.items():
        if n in d and idxset:
            p=d[n]; idx=torch.tensor(sorted(idxset),device=p.device); p.data.view(-1)[idx]=0.0; nz+=len(idxset)
    return nz
print('functions ready | matrices', len(names), '| groups', len(GROUPS))

functions ready | matrices 108 | groups 9


## Step 3 — iterative loop (checkpointed per round, resumable)

In [4]:
CK=f'/kaggle/working/iter_ckpt_{MODEL_TAG}.pt'
if os.path.exists(CK):
    st=torch.load(CK)
    cum={n:set(v) for n,v in st['cum'].items()}; start_round=st['next_round']; rounds_log=st.get('rounds_log',[])
    reapplied=suppress_cumulative(cum)   # restore model to its post-round-(start_round-1) state
    print('resuming from round', start_round, '| cumulative', sum(len(v) for v in cum.values()),
          '| reapplied', reapplied)
else:
    cum={}; start_round=1; rounds_log=[]

model.gradient_checkpointing_enable(); model.config.use_cache=False

for r in range(start_round, N_MAX+1):
    t0=time.time(); new_this_round=0; rd_fp=0; rd_fpng=0
    for gi,grp in enumerate(GROUPS):
        gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
        imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
        for n in gnames:
            fi=coupled_indices_for_matrix(imp_g[n], imp_f[n], imp_p[n])   # authors' exact selection
            ss=coupling_set_sizes(imp_g[n], imp_f[n], imp_p[n]); rd_fp+=ss['n_fp']; rd_fpng+=ss['n_fp_notg']
            if len(fi):
                s=cum.setdefault(n,set()); before=len(s)
                s.update(int(x) for x in fi.tolist()); new_this_round+=len(s)-before
        del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
    nz=suppress_cumulative(cum)   # apply this round's cumulative suppression -> next round scores on it
    tot=sum(len(v) for v in cum.values())
    rounds_log.append({'round':r,'seconds':round(time.time()-t0),'new':int(new_this_round),
                       'cumulative':int(tot),'n_fp':int(rd_fp),'n_fp_notg':int(rd_fpng),'zeroed':int(nz)})
    torch.save({'cum':{n:list(v) for n,v in cum.items()},'rounds_log':rounds_log,'next_round':r+1}, CK)
    json.dump({'model':MODEL_ID,'sampling':'hf_datasets.shuffle(seed).select','rounds':rounds_log},
              open(f'/kaggle/working/iter_summary_{MODEL_TAG}.json','w'), indent=1)
    print(f'  round {r}/{N_MAX} done ({time.time()-t0:.0f}s) | new {new_this_round} | cumulative {tot} | |N_f n N_p| {rd_fp} (clean {rd_fpng}) | zeroed {nz}')
    memline()
    if r in SNAP:
        meta={'model':MODEL_ID,'variant':'iter-SPIN','sampling':'hf_datasets.shuffle(seed).select','round':r,'nsamples':NSAMPLES,'target':TARGET,
              'p':P,'q':Q,'coupled_count':tot,'coupled':{n:sorted(v) for n,v in cum.items()}}
        json.dump(meta, open(f'/kaggle/working/coupled_iter_{MODEL_TAG}_n{r}.json','w'))
        print(f'    snapshot -> coupled_iter_{MODEL_TAG}_n{r}.json ({tot} weights)')

model.gradient_checkpointing_disable()
for p in model.parameters(): p.requires_grad_(False)
print('\nITER-SPIN STEP 1 DONE ('+MODEL_TAG+'). Snapshots:', [f'coupled_iter_{MODEL_TAG}_n{r}.json' for r in SNAP])
print('Next: feed each snapshot to the 7B Step 2 (HSIC+ppl) for the n-ablation curve;')
print('      run Step 3+4 (generate+judge) only for the chosen best n.')
print('Sanity (n=1 == SPIN): for Qwen2-7B, coupled_iter_Qwen2-7B-Instruct_n1.json should match')
print('        the author-sample Step 1 coupled_indices_authorsample.json (1201 weights).')
print('Per-round ablation summary -> iter_summary_'+MODEL_TAG+'.json (new/cumulative/|N_f n N_p| per round).')

  round 1/10 done (710s) | new 645 | cumulative 645 | |N_f n N_p| 782 (clean 438) | zeroed 645
  [] GPU0 12.0/15.6 GPU1 12.1/15.6 GiB free
    snapshot -> coupled_iter_Qwen2.5-3B-Instruct_n1.json (645 weights)
  round 2/10 done (708s) | new 648 | cumulative 1293 | |N_f n N_p| 755 (clean 416) | zeroed 1293
  [] GPU0 12.0/15.6 GPU1 12.1/15.6 GiB free
    snapshot -> coupled_iter_Qwen2.5-3B-Instruct_n2.json (1293 weights)
  round 3/10 done (704s) | new 607 | cumulative 1900 | |N_f n N_p| 666 (clean 339) | zeroed 1900
  [] GPU0 12.0/15.6 GPU1 12.1/15.6 GiB free
    snapshot -> coupled_iter_Qwen2.5-3B-Instruct_n3.json (1900 weights)
  round 4/10 done (707s) | new 555 | cumulative 2455 | |N_f n N_p| 667 (clean 294) | zeroed 2455
  [] GPU0 12.0/15.6 GPU1 12.1/15.6 GiB free
  round 5/10 done (704s) | new 551 | cumulative 3006 | |N_f n N_p| 701 (clean 307) | zeroed 3006
  [] GPU0 12.0/15.6 GPU1 12.1/15.6 GiB free
    snapshot -> coupled_iter_Qwen2.5-3B-Instruct_n5.json (3006 weights)
  round 6/